# Pruning model với Tensorflow API

## Abstract

Tiếp nối chuỗi series nâng cao kiến thức bản thần về ML,DL, bài viết này mình xin phép chia sẻ một bài viết thuộc chủ đề Prunning. Vẫn với lí do lướt Towards Data Science, Medium thì thấy bài viết hay quá nên chia sẻ cùng mọi người

Cùng với việc phát triển mạnh mẽ của công nghệ và dữ liệu đã thúc đẩy DL ngày càng lớn mạnh với những thành tựu đáng kính nể, có những bài toán có độ chính xác vượt xa cả con người. Các mô hình ngày càng lớn mạnh, đi kèm với việc tiêu tốn tài nguyên. KHông nói gì xa, hiện tại khi muốn triển khai DL cho khách hàng, bên cạnh độ chính xác thì luôn phải cân nhắc tới việc tiêu tốn tài nguyên. Làm sao để giải quyết các bài toán lớn nhưng phải phù hợp với tài nguyên hiện tại. Một trong các giải pháp giải quyết vấn đề này phải kể tới kỹ thuật Prunning.

## Pruning là gì?

Nói một cách khái quát, Pruning là một trong những phương pháp đáp ứng việc inference một cách hiệu quả đối với các mô hình có kích thước nhỏ hơn, tiết kiệm bộ nhớ hơn, suy luận nhanh hơn với độ chính xác giảm ít nhất có thể so với mô hình gốc ban đầu

Trong Decision Tree, Pruning là 1 kỹ thuật regularization để tránh Overfitting, trong đó, các leaf node có chung một non-leaf node sẽ được cắt tỉa và non-leaf node đó sẽ trở thành 1 leaf node, với class tương ứng là class chiếm đa số trong số mọi điểm được phân vào node đó

![](image1.png)

Ý tưởng cắt tỉa mạng neural network được lấy cảm hứng từ chính sự cắt tỉa liên kết neural trong não người, nơi các liên kết thần kinh giữa các neural(axon) bị phân rã hoàn toàn và chết đi xảy ra giữa thời thơ ấu và sự khởi đầu của dậy thì. Pruning trong Neural Network chính là loại bỏ các kết nối dư thừa trong kiến trúc mạng. Việc cắt bỏ này thực chất là đưa các giá trị trọng số gần $0$ về $0$ để loại bỏ những kết nối không cần thiết, việc cắt tỉa này sẽ không gây ảnh hưởng nhiều đến quá trình inference.

Có nhiều cách khác nhau để Pruning Model. Có thể cắt tỉa ngay từ đầu một số trọng số ngẫu nhiên, hoặc cũng có thể cắt tỉa khi kết thúc quá trình đào tạo để đơn giản hoá mô hình.

Chắc hẳn sẽ có bạn thắc mắc rằng tại sao một mô hình lại nên được cắt bớt thay vì được khởi tạo với ít tham số hơn từ lúc bắt đầu. Câu trả lời cho câu hỏi này là về bản chất bạn muốn giữa một kiến trúc mô hình tương đối phức tạp để đào tạo , bao quát được dữ liệu. Đồng thời việc tinh chỉnh các lớp, giảm hay tăng kích thước các tính năng là một công việc không đem lại hiệu quả cao. So với đó thì việc Pruning model đơn giản mà mang lại hiệu quả hơn nhiều.

## Pruning cùng Tensorflow

### Giới thiệu tfmot

Tfmot là một công cụ với mục tiêu loại bỏ những weights yếu nhất vào cuối mỗi bước huấn luyện, đồng thời nó cho phép lập trình viên xác định một lịch trình cắt tỉa sẽ tự động xử lý việc loại bỏ các weights.

Bộ lập lịch này tuân theo một lịch trình phân rã đa thức (polynomial decay schedule). Cần truyền vào cho công cụ các tham số như:
- Độ thưa ban đầu (initial sparsity)
- Độ thưa cuối cùng (final sparsity)
- Bước bắt đầu cắt tỉa
- Bước kết thúc cắt tỉa
- Số mũ của phép phân rã (exponent of the polynomial decay) tại mỗi bước, bộ công cụ sẽ loại bỏ đủ weights sao cho độ thưa thớt đạt được là:
$$
S = (S_e - S_0)(\frac{t - t_0}{t_e - t_0})^\alpha
$$

- Trong đó:
    - $S$ là độ thưa thớt
    - $S_e$ là độ thưa thớt cuối cùng
    - $S_0$ là độ thưa thớt ban đầu
    - $t$ là time step hiện tại
    - $t_0$ là time step bắt đầu
    - $\alpha$ là số mũ (mặc định là 3)

Ngoài ra thì các siêu tham số khác cần thay đổi đề tìm ra giá trị tối ưu. Theo lời khuyên của tác giả, cần cắt tỉa từ từ, chút một để mô hình "thích nghi" với việc giảm weights, cũng giống như cắt cây, cắt một lèo thì còn gì đâu...

### Triển khai pruning cùng tfmot với ví dụ đơn giản

Để có thể hình dung và dễ sử dụng tfmot hơn, mình sẽ làm một thí nghiệm nhỏ vừa để hiểu cách sử dụng tfmot, vừa để so sánh việc cắt tỉa và không xem hiệu suất mô hình thay đổi như thế nào.

Tạo dataset:

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_regression

# Parameters of the data-set
n_samples = 10000
n_features = 1000
n_informative = 500
noise = 3

# Create dataset and preprocess it
x, y = make_regression(
    n_samples=n_samples,
    n_features=n_features, n_informative=n_informative,
    noise=noise
)
x = x / abs(x).max(axis=0)
y = y / abs(y).max()
x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.2, random_state=42
)

Tạo model:

In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, ReLU

model = tf.keras.Sequential()
model.add(Dense(1024, kernel_initializer="he_normal", input_dim=n_features))
model.add(ReLU())
model.add(Dense(1024))
model.add(ReLU())
model.add(Dense(1))

Summary mô hình:

![](image2.png)

Với kiến trúc mạng đơn giản như vậy tuy nhiên tổng số lượng params cũng đã lên tới hơn 2 triệu, nói gì là các kiến trúc mạng phức tạp. Vì vậy, mình thử nghiệm việc đào tạo mô hình không Pruning và có Pruning xem có thay đổi đáng kể hiệu suất mô hình hay không

Training mô hình không sử dụng Pruning:

In [8]:
model.compile(
    loss="mse",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001)
)
history = model.fit(
    x_train,
    y_train,
    validation_data = (x_val, y_val),
    epochs=200,
    batch_size=1024,
    verbose=1
)

Epoch 1/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 56ms/step - loss: 1.9295 - val_loss: 0.0811
Epoch 2/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.2714 - val_loss: 0.1245
Epoch 3/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 0.0664 - val_loss: 0.0774
Epoch 4/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 0.0730 - val_loss: 0.0671
Epoch 5/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 0.0449 - val_loss: 0.0428
Epoch 6/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 0.0335 - val_loss: 0.0408
Epoch 7/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 0.0297 - val_loss: 0.0355
Epoch 8/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 0.0218 - val_loss: 0.0296
Epoch 9/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 0.0156 - val_loss: 0.0251
Epoch 10/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 0.0108 - val_loss: 0.0198
Epoch 11/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 0.0069 - val_loss: 0.0156
Epoch 12/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 0.0045 - val_lo

![](image3.png)

Training mô hình sử dụng Pruning với công cụ tfmot:

In [ ]:
import tensorflow_model_optimization as tfmot

initial_sparsity = 0.0
final_sparsity = 0.75
begin_step = 1000
end_step = 5000
pruning_params = {
    'pruning_schedule': tfmot.sparsity.PolynomialDecay(
        initial_sparsity=initial_sparsity,
        final_sparsity=final_sparsity,
        begin_step=begin_step,
        end_step=end_step
    )
}
model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)
pruning_callback = tfmot.sparsity.keras.UpdatePruningStep()

Ở đây sử dụng tfmot như 1 callback, giống learning rate scheduler và early stopping

In [ ]:
model.compile(
    loss="mse",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001)
)
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=200,
    batch_size=1024,
    callbacks=pruning_callback,
    verbose=1
)

![](image4.png)

#### loss ↓ rất thấp, val_loss đứng yên $\rightarrow$ Capacity đã chạm trần

Đã có sự chênh lệch tuy nhiên khi sử dụng Pruning thì độ chính xác giảm đi không nhiều, val_loss vẫn ở mức chấp nhận được

## Kết luận

#### Không pruning
- loss → ~1e-16 (gần 0 tuyệt đối)
- val_loss → ~0.0085

$\rightarrow$ Model quá mạnh, fit train cực tốt

$\rightarrow$ Nhưng generalization không cải thiện thêm

#### Có pruning
- loss → ~1e-5
- val_loss → ~0.0041

$\rightarrow$ Model không fit hoàn hảo train

$\rightarrow$ Nhưng generalize tốt hơn

$\rightarrow$ Đây là ví dụ bias–variance tradeoff rất sạch

## Pruning là val_loss không cần fit sát train $\rightarrow$ chấp nhận performance giảm một chút để đổi lấy model nhỏ hơn (ít weight hơn)

hoặc

## Pruning không làm model “ngu đi”, nó làm model “bớt tham”.

Với cá nhân mình thấy Pruning là mọt phương pháp đang khá được để tâm tới do tính hữu dụng của nó trong việc “làm nhẹ” mô hình. MÌnh đã thử nghiệm một vài bái toán cùng team và nghe một vài buổi seminar nới về Pruning thì thấy kết quả của các tác giả thử nghiệm đem lại hiệu quả tương đối bất ngờ.

Song song với việc phát triển phần cứng thì chúng ta cũng cần có nhwung phương pháp xoa dịu mô hình nhẹ xuống để phần cứng hay tài chính, thời gian còn theo kịp =)))